In [1]:
from unsloth import FastLanguageModel
import json

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/ubuntu/miniconda/envs/unsloth_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.5.1 with CUDA 1201 (you have 2.8.0+cu128)
    Python  3.11.10 (you have 3.11.13)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


🦥 Unsloth Zoo will now patch everything to make training faster!


In [7]:
max_seq_length = None
load_in_4bit = True
dtype = None 
model_name_str = 'end_to_end_v4_val_gt'
model_name = f'/data2/finetuned_llms/{model_name_str}'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name, # YOUR MODEL YOU USED FOR TRAINING
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

from datasets import load_dataset

base_path = '/data2/jsonl/val.jsonl'

data_files = {
    'val': base_path
}

dataset = load_dataset(
    "json",
    data_files=data_files
)

print(dataset['val']['text'][0])


==((====))==  Unsloth 2025.8.1: Fast Llama patching. Transformers: 4.55.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.278 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
<|start_header_id|>user<|end_header_id|>

You are helping decode speech from neural activity to help restore communication for a paralyzed patient. For each time bin of neural activity, a neural network model provides the 10 most probable tokens. Tokens consist of ARPAbet phonemes and the space character, denoted as <>. On each line, the the top 10 tokens are listed in order, from most to least likely. Each separate line represents the model output for a given non-overlapping neural time bin, starting from the beginning of the text. Since the mod

In [9]:
batch_size = 1
split = "val"

skip_even = True

last_lines_val = []
n = len(dataset[split])

for batch_idx in range(0, 880):
    
    if skip_even:
        if batch_idx % 2 == 0:
            continue
    
    if batch_idx % 80 == 0:
        print(batch_idx)

    # Grab a batch of texts
    batch = dataset[split][batch_idx]
    val_texts = batch["text"]
    
    # Tokenize the batch
    inputs = tokenizer(
        val_texts,
        return_tensors='pt',
        padding=True,
        truncation=True
    ).to('cuda')

    # Generate
    outputs = model.generate(
        **inputs,
        use_cache=True, 
        max_new_tokens=400
    )

    # Decode all outputs
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    # Keep just the last non-empty line from each sequence
    for seq in decoded:
        lines = [l.strip() for l in seq.splitlines() if l.strip()]
        last_line = lines[-1] if lines else ""
        last_lines_val.append(last_line)
        

with open(f'/data2/jsonl/val_sents_{model_name_str}.json', "w") as f:
    json.dump(last_lines_val, f)

In [14]:
import json
from cer_wer import _cer_and_wer
with open("/data2/jsonl/val_ground_truth.json", "r") as f:
    val_gt = json.load(f)
    
if skip_even:
    val_gt = val_gt[1::]
_cer_and_wer(last_lines_val, val_gt)

(np.float64(0.1903032504085709), np.float64(0.2624113475177305))